# GAT-v4 | Phase 6 — Feature Tensor Assembly

**Covers plan section 8**

Builds the per-fold feature tensors consumed by the GAT model.
All normalisation is **fold-aware** — statistics computed on training tumors only.

### Per-gene features (3D per gene per sample)
| Dim | Feature | Normalisation |
|-----|---------|---------------|
| 0 | RNA expression (VST) | z-score using training tumor mean + std |
| 1 | CNV (GISTIC2) | centred on training tumor mean |
| 2 | Node degree | fixed per fold edge set, scaled to [0,1] |

### Per-patient clinical features (3D)
| Dim | Feature | Normalisation |
|-----|---------|---------------|
| 0 | Age | z-score on training tumors |
| 1 | Stage | ordinal / 4 → [0,1] |
| 2 | Grade | ordinal / 4 → [0,1] (or 0.5 if missing) |

### Shapes saved per fold
```
feature_tensor.npy   — (n_tumor_samples, n_ppi_nodes, 3)
clinical_tensor.npy  — (n_tumor_samples, 3)
sample_ids.json      — ordered list of sample IDs matching tensor rows
survival_labels.npy  — (n_tumor_samples, 2) columns: [os_months, event]
```

---
| Cell | What |
|------|------|
| 1 | Imports & paths |
| 2 | Load all inputs |
| 3 | Build per-fold tensors (main loop) |
| 4 | Validate one fold |
| 5 | Save + audit YAML |

---
## Cell 1 — Imports & Paths

In [7]:
import sys, json, yaml, hashlib, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script")
KIRC_ROOT   = PROJECT_ROOT / "KIRC"
DATA_PROC    = KIRC_ROOT / "data" / "processed"
DATA_SPLITS  = KIRC_ROOT / "data" / "splits"
AUDIT_DIR    = KIRC_ROOT / "audit"
# ─────────────────────────────────────────────────────────────────────────────

if str(KIRC_ROOT) not in sys.path:
    sys.path.insert(0, str(KIRC_ROOT))
from src.utils.logger import get_logger
log = get_logger('phase6_features')
print("Imports OK.")

C:\Users\jiyad\Desktop\PROJECT\KIRC\script\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-06 23:16:07 | INFO     | phase6_features | Logger initialised. Log file → C:\Users\jiyad\OneDrive\Desktop\PROJECT\gat-v4\results\logs\phase6_features_20260706_231607.log


Imports OK.


In [8]:
# Fix stage encoding and rewrite clinical_clean.csv
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PROC    = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\processed")
CLIN_PATIENT = Path(r"C:\Users\jiyad\Desktop\PROJECT\KIRC\raw_data\CNV\data_clinical_patient.txt")

raw = pd.read_csv(CLIN_PATIENT, sep='\t', comment='#', dtype=str)
raw.columns = raw.columns.str.strip().str.upper()
raw['PATIENT_ID'] = raw['PATIENT_ID'].str.lstrip('#').str.strip()

def encode_stage_fixed(val):
    """Longest-match-first — IV before I, III before II."""
    if pd.isna(val) or str(val).strip() in ['[Discrepancy]', '[Not Available]', '']:
        return np.nan
    v = str(val).upper().strip()
    if 'IV'  in v: return 4
    if 'III' in v: return 3
    if 'II'  in v: return 2
    if 'I'   in v: return 1
    return np.nan

raw['stage_fixed'] = raw['AJCC_PATHOLOGIC_TUMOR_STAGE'].apply(encode_stage_fixed)

print("Fixed stage distribution:")
print(raw['stage_fixed'].value_counts(dropna=False).sort_index().to_string())
print(f"\n[Discrepancy] rows → NaN (will be dropped): "
      f"{raw['stage_fixed'].isna().sum()}")

# Patch clinical_clean.csv
clin = pd.read_csv(DATA_PROC / 'clinical_clean.csv')
stage_map = raw.dropna(subset=['stage_fixed']).set_index('PATIENT_ID')['stage_fixed'].to_dict()
clin['stage'] = clin['patient_id'].map(stage_map)

print(f"\nPatched stage distribution in clinical_clean:")
print(clin['stage'].value_counts(dropna=False).sort_index().to_string())

# Drop the 3 [Discrepancy] patients (no valid stage)
n_before = len(clin)
clin = clin.dropna(subset=['stage']).copy()
print(f"\nPatients: {n_before} → {len(clin)}  (dropped {n_before - len(clin)} [Discrepancy])")

clin.to_csv(DATA_PROC / 'clinical_clean.csv', index=False)
print("\nclinical_clean.csv saved. Re-run Phase 6 Cells 3 and 4.")

Fixed stage distribution:
stage_fixed
1    249
2     54
3    126
4     83

[Discrepancy] rows → NaN (will be dropped): 0

Patched stage distribution in clinical_clean:
stage
1    246
2     54
3    125
4     82

Patients: 507 → 507  (dropped 0 [Discrepancy])

clinical_clean.csv saved. Re-run Phase 6 Cells 3 and 4.


---
## Cell 2 — Load All Inputs

Loads everything needed before the per-fold loop:
- VST matrix (DEG rows)
- CNV aligned matrix
- Clinical clean
- PPI gene index (node ordering)
- CV splits
- Sample alignment

In [9]:
t0 = time.time()

# ── VST matrix (genes x samples) ─────────────────────────────────────────────
print("Loading VST matrix...")
vst = pd.read_csv(DATA_PROC / 'vst_matrix.csv', index_col=0)
print(f"  VST shape : {vst.shape}")

# ── CNV aligned (genes x samples) ────────────────────────────────────────────
print("Loading CNV aligned...")
cnv = pd.read_csv(DATA_PROC / 'cnv_aligned.csv', index_col=0)
print(f"  CNV shape : {cnv.shape}")

# ── Clinical ──────────────────────────────────────────────────────────────────
clin = pd.read_csv(DATA_PROC / 'clinical_clean.csv')
clin = clin.set_index('patient_id')
print(f"  Clinical  : {clin.shape}  cols={list(clin.columns)}")

# ── PPI gene index (defines node ordering for the graph) ─────────────────────
gene_idx_df  = pd.read_csv(DATA_PROC / 'ppi_gene_index.csv')
ppi_genes    = gene_idx_df['gene_symbol'].tolist()  # ordered list
gene_to_idx  = {g: i for i, g in enumerate(ppi_genes)}
n_nodes      = len(ppi_genes)
print(f"  PPI nodes : {n_nodes:,}")

# ── CV splits ────────────────────────────────────────────────────────────────
with open(DATA_SPLITS / 'cv_folds_repeated_5x3.json') as f:
    cv_splits = json.load(f)
print(f"  CV folds  : {len(cv_splits)}")

# ── Sample alignment ──────────────────────────────────────────────────────────
align = pd.read_csv(DATA_PROC / 'sample_alignment.csv')
# Patient → sample ID map (tumor only, included only)
align_tumor  = align[(align['tissue_type']=='tumor') & align['include']]
pat_to_sids  = align_tumor.groupby('patient_id')['sample_id'].apply(list).to_dict()
# Sample → patient map
sid_to_pat   = dict(zip(align_tumor['sample_id'], align_tumor['patient_id']))

print(f"\nAll inputs loaded in {time.time()-t0:.0f}s")

# ── Gene availability checks ──────────────────────────────────────────────────
ppi_in_vst = sum(1 for g in ppi_genes if g in vst.index)
ppi_in_cnv = sum(1 for g in ppi_genes if g in cnv.index)
print(f"\nPPI genes in VST : {ppi_in_vst:,} / {n_nodes:,}  ({ppi_in_vst/n_nodes*100:.1f}%)")
print(f"PPI genes in CNV : {ppi_in_cnv:,} / {n_nodes:,}  ({ppi_in_cnv/n_nodes*100:.1f}%)")

# Pre-index VST and CNV to PPI gene order for fast numpy slicing
# Shape: (n_nodes, n_samples) — NaN for genes not in the matrix
vst_ppi = vst.reindex(ppi_genes)      # (n_nodes x all_samples)
cnv_ppi = cnv.reindex(ppi_genes)      # (n_nodes x tumor_samples)

log.info(f"Inputs loaded: VST {vst.shape}, CNV {cnv.shape}, nodes {n_nodes}")
print("\nCell 2 done.")

Loading VST matrix...
  VST shape : (9997, 569)
Loading CNV aligned...
  CNV shape : (9997, 560)
  Clinical  : (507, 5)  cols=['event', 'os_months', 'age', 'stage', 'grade']
  PPI nodes : 4,414
  CV folds  : 15

All inputs loaded in 5s

PPI genes in VST : 4,414 / 4,414  (100.0%)
PPI genes in CNV : 4,414 / 4,414  (100.0%)


2026-07-06 23:16:13 | INFO     | phase6_features | Inputs loaded: VST (9997, 569), CNV (9997, 560), nodes 4414



Cell 2 done.


---
## Cell 3 — Build Per-Fold Tensors

For each of the 15 folds:

**RNA feature (fold-aware z-score):**
- Mean and std computed on **training tumor samples only**
- Applied to all included tumor samples (train + val + test)
- Missing genes (not in VST) → z-score = 0 (mean-imputed)

**CNV feature (fold-aware centering):**
- Mean computed on **training tumor samples only**
- Applied to all included tumor samples
- CNV column names are short barcodes — matched to full sample IDs via patient ID

**Degree feature:**
- Node degree in the **fold-specific pruned edge set** (after |r| >= 0.3)
- Scaled to [0, 1] by dividing by max degree in that fold

**Runtime: ~5–15 min total**

In [10]:
# ── CNV column matching helper ────────────────────────────────────────────────
# CNV columns are short barcodes: TCGA-XX-YYYY-ZZ
# Full sample IDs:                TCGA-XX-YYYY-ZZA-01R-...
# Match via patient ID (first 3 fields)

cnv_col_to_pat = {col: '-'.join(col.split('-')[:3]) for col in cnv_ppi.columns}
pat_to_cnv_col = {}  # patient_id → CNV column name
for col, pat in cnv_col_to_pat.items():
    if pat not in pat_to_cnv_col:  # take first match
        pat_to_cnv_col[pat] = col

print(f"CNV patient coverage : {len(pat_to_cnv_col):,} patients have CNV columns")

# ── Main per-fold loop ────────────────────────────────────────────────────────
fold_meta = []

for fold_key, fold in tqdm(cv_splits.items(), desc='Building tensors'):
    fold_dir = DATA_SPLITS / fold_key

    # ── 1. Identify all tumor sample IDs for this fold (train+val+test) ──────
    all_fold_pats  = fold['train_patients'] + fold['val_patients'] + fold['test_patients']
    train_pats     = set(fold['train_patients'])

    fold_sids = []         # full sample IDs in this fold
    fold_pats = []         # corresponding patient IDs
    for pat in all_fold_pats:
        for sid in pat_to_sids.get(pat, []):
            if sid in vst_ppi.columns:
                fold_sids.append(sid)
                fold_pats.append(pat)

    if not fold_sids:
        log.warning(f"{fold_key}: no samples found")
        continue

    n_samp = len(fold_sids)

    # ── 2. Training tumor VST (for computing normalisation stats) ────────────
    train_sids_fold = [s for s, p in zip(fold_sids, fold_pats) if p in train_pats]

    # VST block: (n_nodes, n_fold_samples)
    vst_block = vst_ppi[fold_sids].values.astype(np.float32)   # (nodes, samps)
    # Training columns only for stat computation
    train_col_mask = np.array([s in set(train_sids_fold) for s in fold_sids])
    vst_train_only = vst_block[:, train_col_mask]               # (nodes, n_train)

    # Mean + std on training tumors (ignore NaN)
    train_mean = np.nanmean(vst_train_only, axis=1)             # (nodes,)
    train_std  = np.nanstd(vst_train_only,  axis=1)             # (nodes,)
    train_std  = np.where(train_std < 1e-6, 1.0, train_std)    # avoid /0

    # z-score ALL fold samples using training stats
    rna_z = (vst_block - train_mean[:, None]) / train_std[:, None]   # (nodes, samps)
    # Fill NaN (genes missing from VST) with 0
    rna_z = np.nan_to_num(rna_z, nan=0.0)
    # Clip extreme z-scores (|z| > 10 are likely artefacts)
    rna_z = np.clip(rna_z, -10.0, 10.0)

    # ── 3. CNV feature (fold-aware centering) ────────────────────────────────
    cnv_block = np.zeros((n_nodes, n_samp), dtype=np.float32)
    for j, (sid, pat) in enumerate(zip(fold_sids, fold_pats)):
        cnv_col = pat_to_cnv_col.get(pat)
        if cnv_col is not None and cnv_col in cnv_ppi.columns:
            cnv_block[:, j] = cnv_ppi[cnv_col].fillna(0).values.astype(np.float32)
        # else: remains 0 (no CNV = diploid assumption)

    # Centre on training tumor mean
    cnv_train_only = cnv_block[:, train_col_mask]
    cnv_train_mean = cnv_train_only.mean(axis=1)                # (nodes,)
    cnv_centered   = cnv_block - cnv_train_mean[:, None]        # (nodes, samps)

    # ── 4. Degree feature (fold-specific edge set) ───────────────────────────
    edge_index = np.load(fold_dir / 'edge_index.npy')           # (2, n_edges)
    degree     = np.zeros(n_nodes, dtype=np.float32)
    for node_idx in edge_index[0]:
        degree[node_idx] += 1
    for node_idx in edge_index[1]:
        degree[node_idx] += 1
    max_deg = degree.max()
    degree_scaled = degree / max_deg if max_deg > 0 else degree  # [0, 1]
    # Broadcast to all samples: (n_nodes,) → (n_nodes, n_samp)
    degree_block = np.broadcast_to(degree_scaled[:, None],
                                   (n_nodes, n_samp)).copy().astype(np.float32)

    # ── 5. Stack gene features: (n_nodes, n_samp, 3) ─────────────────────────
    # Axes: [node, sample, feature_dim]
    feat = np.stack([rna_z, cnv_centered, degree_block], axis=2)  # (nodes, samps, 3)
    # Transpose to (n_samp, n_nodes, 3) — standard GNN input convention
    feat = feat.transpose(1, 0, 2)                                 # (samps, nodes, 3)

    # ── 6. Clinical feature tensor: (n_samp, 3) ──────────────────────────────
    train_clin = clin.loc[clin.index.isin(train_pats)]

    # Age z-score — fallback if missing
    if 'age' in clin.columns and train_clin['age'].notna().sum() > 5:
        age_mean = float(train_clin['age'].mean())
        age_std  = float(train_clin['age'].std())
        age_std  = age_std if age_std > 1e-6 else 1.0
    else:
        age_mean, age_std = 60.0, 10.0

    # Stage: check if it actually varies
    if 'stage' in clin.columns:
        stage_vals = train_clin['stage'].dropna()
        stage_median = float(stage_vals.median()) if len(stage_vals) > 0 else 2.0
    else:
        stage_median = 2.0

    # Grade: may be entirely missing in cBioPortal KIRC download
    if 'grade' in clin.columns:
        grade_vals = train_clin['grade'].dropna()
        grade_median = float(grade_vals.median()) if len(grade_vals) > 0 else 2.0
        has_grade_data = len(grade_vals) > 0
    else:
        grade_median, has_grade_data = 2.0, False

    clin_arr = np.zeros((n_samp, 3), dtype=np.float32)
    for j, (sid, pat) in enumerate(zip(fold_sids, fold_pats)):
        if pat in clin.index:
            row = clin.loc[pat]

            # Age
            age_raw = row.get('age', age_mean)
            age_val = float(age_raw) if pd.notna(age_raw) else age_mean
            age_z   = np.clip((age_val - age_mean) / age_std, -4, 4)

            # Stage — fill NaN with training median
            stage_raw = row.get('stage', stage_median)
            stage_val = float(stage_raw) if pd.notna(stage_raw) else stage_median
            stage_s   = np.clip(stage_val / 4.0, 0, 1)

            # Grade — fill NaN with training median (or 0.5 if no grade data)
            grade_raw = row.get('grade', grade_median) if has_grade_data else grade_median
            grade_val = float(grade_raw) if pd.notna(grade_raw) else grade_median
            grade_s   = np.clip(grade_val / 4.0, 0, 1)

            clin_arr[j] = [age_z, stage_s, grade_s]
        else:
            # No clinical data — use neutral values (0=mean age, 0.5=stage II, 0.5=grade II)
            clin_arr[j] = [0.0, 0.5, 0.5]

    # ── 7. Survival labels: (n_samp, 2) → [os_months, event] ─────────────────
    surv_arr = np.zeros((n_samp, 2), dtype=np.float32)
    for j, (sid, pat) in enumerate(zip(fold_sids, fold_pats)):
        if pat in clin.index:
            row = clin.loc[pat]
            surv_arr[j, 0] = float(row.get('os_months', 0))
            surv_arr[j, 1] = float(row.get('event',     0))

    # ── 8. Save ───────────────────────────────────────────────────────────────
    np.save(fold_dir / 'feature_tensor.npy',   feat)       # (n_samp, n_nodes, 3)
    np.save(fold_dir / 'clinical_tensor.npy',  clin_arr)   # (n_samp, 3)
    np.save(fold_dir / 'survival_labels.npy',  surv_arr)   # (n_samp, 2)

    with open(fold_dir / 'sample_ids.json', 'w') as f:
        json.dump(fold_sids, f)

    # Split index arrays (for training loader)
    train_idx = [i for i, p in enumerate(fold_pats) if p in train_pats]
    val_idx   = [i for i, p in enumerate(fold_pats)
                 if p in set(fold['val_patients'])]
    test_idx  = [i for i, p in enumerate(fold_pats)
                 if p in set(fold['test_patients'])]

    with open(fold_dir / 'split_indices.json', 'w') as f:
        json.dump({'train': train_idx, 'val': val_idx, 'test': test_idx}, f)

    fold_meta.append({
        'fold_key'     : fold_key,
        'n_samples'    : n_samp,
        'n_train'      : len(train_idx),
        'n_val'        : len(val_idx),
        'n_test'       : len(test_idx),
        'feat_shape'   : list(feat.shape),
        'clin_shape'   : list(clin_arr.shape),
        'rna_z_mean'   : round(float(rna_z.mean()), 4),
        'rna_z_std'    : round(float(rna_z.std()),  4),
        'cnv_mean'     : round(float(cnv_centered.mean()), 4),
        'degree_max'   : round(float(max_deg), 1),
    })

print(f"\nTensors built for {len(fold_meta)} folds")
print()
print(f"{'Fold':<25} {'N_samp':>7} {'Train':>6} {'Val':>5} {'Test':>5} {'Feat shape'}")
for m in fold_meta:
    print(f"{m['fold_key']:<25} {m['n_samples']:>7} {m['n_train']:>6} "
          f"{m['n_val']:>5} {m['n_test']:>5} {m['feat_shape']}")

log.info(f"Feature tensors built for {len(fold_meta)} folds")
print("\nCell 3 done.")

CNV patient coverage : 493 patients have CNV columns



Building tensors: 100%|██████████| 15/15 [00:20<00:00,  1.33s/it]
2026-07-06 23:16:33 | INFO     | phase6_features | Feature tensors built for 15 folds



Tensors built for 15 folds

Fold                       N_samp  Train   Val  Test Feat shape
seed42_fold0                  500    361    40    99 [500, 4414, 3]
seed42_fold1                  500    360    40   100 [500, 4414, 3]
seed42_fold2                  500    360    40   100 [500, 4414, 3]
seed42_fold3                  500    362    40    98 [500, 4414, 3]
seed42_fold4                  500    357    40   103 [500, 4414, 3]
seed123_fold0                 500    361    40    99 [500, 4414, 3]
seed123_fold1                 500    360    42    98 [500, 4414, 3]
seed123_fold2                 500    357    40   103 [500, 4414, 3]
seed123_fold3                 500    360    40   100 [500, 4414, 3]
seed123_fold4                 500    360    40   100 [500, 4414, 3]
seed456_fold0                 500    361    40    99 [500, 4414, 3]
seed456_fold1                 500    361    40    99 [500, 4414, 3]
seed456_fold2                 500    360    40   100 [500, 4414, 3]
seed456_fold3          

---
## Cell 4 — Validate One Fold

Spot-checks the first fold for shape correctness, value ranges, and leakage.
**Leakage check:** test set RNA z-scores must NOT be zero-mean (they would be if
test samples were included in the normalisation — would indicate leakage).

In [11]:
# Load the first fold back from disk
check_key = list(cv_splits.keys())[0]
check_dir = DATA_SPLITS / check_key

feat_loaded  = np.load(check_dir / 'feature_tensor.npy')   # (n_samp, n_nodes, 3)
clin_loaded  = np.load(check_dir / 'clinical_tensor.npy')  # (n_samp, 3)
surv_loaded  = np.load(check_dir / 'survival_labels.npy')  # (n_samp, 2)
ei_loaded    = np.load(check_dir / 'edge_index.npy')        # (2, n_edges)
ew_loaded    = np.load(check_dir / 'edge_weight.npy')       # (n_edges,)

with open(check_dir / 'split_indices.json') as f:
    split_idx = json.load(f)
with open(check_dir / 'sample_ids.json') as f:
    sids_loaded = json.load(f)

print(f"Validating fold: {check_key}")
print(f"{'='*45}")
print(f"feature_tensor  : {feat_loaded.shape}   expected ({len(sids_loaded)}, {n_nodes}, 3)")
print(f"clinical_tensor : {clin_loaded.shape}  expected ({len(sids_loaded)}, 3)")
print(f"survival_labels : {surv_loaded.shape}  expected ({len(sids_loaded)}, 2)")
print(f"edge_index      : {ei_loaded.shape}")
print(f"edge_weight     : {ew_loaded.shape}")
print()

checks = {}

# Shape checks
checks['feature_tensor_shape'] = feat_loaded.shape[1] == n_nodes and feat_loaded.shape[2] == 3
checks['clinical_tensor_shape'] = clin_loaded.shape[1] == 3
checks['survival_shape'] = surv_loaded.shape[1] == 2
checks['edge_index_shape'] = ei_loaded.shape[0] == 2

# Value range checks
rna_feat  = feat_loaded[:, :, 0]   # RNA z-scores
cnv_feat  = feat_loaded[:, :, 1]   # CNV centered
deg_feat  = feat_loaded[:, :, 2]   # degree [0,1]

checks['rna_z_clipped']    = float(rna_feat.max()) <= 10.0 and float(rna_feat.min()) >= -10.0
checks['degree_in_0_1']    = float(deg_feat.min()) >= 0.0  and float(deg_feat.max()) <= 1.0
checks['edge_weight_0_1']  = float(ew_loaded.min()) >= 0.0 and float(ew_loaded.max()) <= 1.0
checks['no_nan_features']  = not np.isnan(feat_loaded).any()
checks['no_nan_clinical']  = not np.isnan(clin_loaded).any()
checks['surv_events_01']   = set(np.unique(surv_loaded[:, 1]).astype(int)).issubset({0, 1})

# Leakage check: train z-scores should be ~zero-mean; test z-scores should NOT
train_idx_c = split_idx['train']
test_idx_c  = split_idx['test']
train_rna_mean = float(rna_feat[train_idx_c].mean())
test_rna_mean  = float(rna_feat[test_idx_c].mean())
# Training should be close to 0 (normalised), test should deviate
checks['train_rna_near_zero'] = abs(train_rna_mean) < 0.05
# Test mean should not be suspiciously close to 0 (would indicate leakage)
# Note: it may be close to 0 by chance — flag but don't fail

print(f"Feature value ranges:")
print(f"  RNA z-score  : [{rna_feat.min():.2f}, {rna_feat.max():.2f}]  mean={rna_feat.mean():.4f}")
print(f"  CNV centered : [{cnv_feat.min():.2f}, {cnv_feat.max():.2f}]  mean={cnv_feat.mean():.4f}")
print(f"  Degree [0,1] : [{deg_feat.min():.3f}, {deg_feat.max():.3f}]  mean={deg_feat.mean():.4f}")
print(f"  Edge weights : [{ew_loaded.min():.3f}, {ew_loaded.max():.3f}]  mean={ew_loaded.mean():.3f}")
print()
print(f"Leakage check:")
print(f"  Train RNA z-score mean : {train_rna_mean:.5f}  (expected ~0.0)")
print(f"  Test  RNA z-score mean : {test_rna_mean:.5f}   (expected != 0.0 if no leakage)")
print()
print(f"Clinical tensor:")
print(f"  Age z    : [{clin_loaded[:,0].min():.2f}, {clin_loaded[:,0].max():.2f}]")
print(f"  Stage    : [{clin_loaded[:,1].min():.2f}, {clin_loaded[:,1].max():.2f}]")
print(f"  Grade    : [{clin_loaded[:,2].min():.2f}, {clin_loaded[:,2].max():.2f}]")
print()
print(f"Survival:")
print(f"  OS months: [{surv_loaded[:,0].min():.1f}, {surv_loaded[:,0].max():.1f}]")
print(f"  Events   : {int(surv_loaded[:,1].sum())} / {len(surv_loaded)} samples")
print()
print(f"Split sizes:")
print(f"  Train : {len(train_idx_c)}  Val : {len(split_idx['val'])}  Test : {len(test_idx_c)}")
print()

n_pass = sum(checks.values())
for name, ok in checks.items():
    print(f"  {'OK  ' if ok else 'FAIL'} {name}")
print(f"\nValidation: {n_pass}/{len(checks)} checks passed")

if n_pass == len(checks):
    print("All checks passed. Feature tensors are valid.")
else:
    print("Fix failed checks before training.")

Validating fold: seed42_fold0
feature_tensor  : (500, 4414, 3)   expected (500, 4414, 3)
clinical_tensor : (500, 3)  expected (500, 3)
survival_labels : (500, 2)  expected (500, 2)
edge_index      : (2, 14652)
edge_weight     : (14652,)

Feature value ranges:
  RNA z-score  : [-9.65, 10.00]  mean=0.0052
  CNV centered : [-2.78, 2.96]  mean=0.0098
  Degree [0,1] : [0.000, 1.000]  mean=0.0325
  Edge weights : [0.300, 0.978]  mean=0.572

Leakage check:
  Train RNA z-score mean : -0.00003  (expected ~0.0)
  Test  RNA z-score mean : 0.01471   (expected != 0.0 if no leakage)

Clinical tensor:
  Age z    : [-2.58, 2.47]
  Stage    : [0.25, 1.00]
  Grade    : [0.25, 1.00]

Survival:
  OS months: [0.0, 149.1]
  Events   : 160 / 500 samples

Split sizes:
  Train : 361  Val : 40  Test : 99

  OK   feature_tensor_shape
  OK   clinical_tensor_shape
  OK   survival_shape
  OK   edge_index_shape
  OK   rna_z_clipped
  OK   degree_in_0_1
  OK   edge_weight_0_1
  OK   no_nan_features
  OK   no_nan_clin

---
## Cell 5 — Save Fold Meta + Audit YAML

In [12]:
def file_md5(path):
    h = hashlib.md5()
    with open(path,'rb') as f:
        while d := f.read(8192): h.update(d)
    return h.hexdigest()

# Save fold-level metadata as CSV for reference
meta_df   = pd.DataFrame(fold_meta)
meta_path = DATA_SPLITS / 'feature_tensor_meta.csv'
meta_df.to_csv(meta_path, index=False)
print(f"Fold meta -> {meta_path}")

# Aggregate stats
mean_rna_z_std = float(meta_df['rna_z_std'].mean())
mean_n_samp    = float(meta_df['n_samples'].mean())

audit = {
    'phase'    : 'Phase 6 — Feature Tensor Assembly',
    'timestamp': datetime.now().isoformat(),

    'feature_design': {
        'dim_0_rna'    : 'VST z-score (training tumor mean+std per fold)',
        'dim_1_cnv'    : 'GISTIC2 centered on training tumor mean per fold',
        'dim_2_degree' : 'Node degree in fold-specific pruned edge set, scaled [0,1]',
        'clinical_0'   : 'Age z-score (training tumor stats)',
        'clinical_1'   : 'Stage ordinal /4 → [0.25, 1.0]',
        'clinical_2'   : 'Grade ordinal /4 → [0.25, 1.0]',
        'leakage_guard': 'ALL normalisation stats from training tumor only',
    },

    'tensor_shapes': {
        'feature_tensor' : f'(n_samples, {n_nodes}, 3)',
        'clinical_tensor': '(n_samples, 3)',
        'survival_labels': '(n_samples, 2)  [os_months, event]',
        'edge_index'     : '(2, n_fold_edges)',
        'edge_weight'    : '(n_fold_edges,)',
    },

    'fold_summary': {
        'n_folds'         : len(fold_meta),
        'mean_n_samples'  : round(mean_n_samp, 1),
        'mean_rna_z_std'  : round(mean_rna_z_std, 4),
        'validation_pass' : n_pass == len(checks),
    },

    'per_fold': [
        {k: v for k, v in m.items() if k != 'feat_shape'}
        for m in fold_meta
    ],

    'next_phase' : 'Phase 7 — Model Training (07_model_training.ipynb)',
    'status'     : 'COMPLETE' if n_pass == len(checks) else 'REVIEW_NEEDED',
}

audit_path = AUDIT_DIR / 'phase6_feature_tensors.yaml'
with open(audit_path, 'w', encoding='utf-8') as f:
    yaml.dump(audit, f, default_flow_style=False, sort_keys=False)
print(f"Audit YAML -> {audit_path}")

print()
print("=" * 55)
print("PHASE 6 COMPLETE")
print("=" * 55)
print(f"  Folds built      : {len(fold_meta)}")
print(f"  Feature shape    : (n_samples, {n_nodes}, 3)")
print(f"  Clinical shape   : (n_samples, 3)")
print(f"  Mean samples/fold: {mean_n_samp:.0f}")
print(f"  Validation       : {n_pass}/{len(checks)} checks passed")
print()
print("Next: 07_model_training.ipynb — GAT-v4 training across 15 folds")

Fold meta -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\data\splits\feature_tensor_meta.csv
Audit YAML -> C:\Users\jiyad\Desktop\PROJECT\KIRC\script\KIRC\audit\phase6_feature_tensors.yaml

PHASE 6 COMPLETE
  Folds built      : 15
  Feature shape    : (n_samples, 4414, 3)
  Clinical shape   : (n_samples, 3)
  Mean samples/fold: 500
  Validation       : 11/11 checks passed

Next: 07_model_training.ipynb — GAT-v4 training across 15 folds
